In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from FEATURES.featuresV2 import *
from PRODUCTION.calculateEVS import *
from PRODUCTION.helperFunctions import *
from PRODUCTION.teamInfo import teamStarPlayer, projectedStartingFive, mainStartingFive

/Users/alexgonzalez/Documents/NBA-Prop-Predictor/nba_model/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Update projected starting lineups

In [2]:
from MODELS.scrapStarting import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

Successfully updated /Users/alexgonzalez/Documents/NBA-Prop-Predictor/PRODUCTION/teamInfo.py
Updated 18 teams with confirmed lineups


### Load Model

### Load Player Data and Bookmaker Data

In [2]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')

usData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_{today}.csv')
dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')

dfsData.head()

,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE
0,Underdog,player_points,Donovan Mitchell,Over,28.5,-137,2025-12-04,2025-12-03T23:44:28Z
1,Underdog,player_points,Donovan Mitchell,Under,28.5,-137,2025-12-04,2025-12-03T23:44:28Z
2,Underdog,player_points,Darius Garland,Over,18.5,-137,2025-12-04,2025-12-03T23:44:28Z
3,Underdog,player_points,Darius Garland,Under,18.5,-137,2025-12-04,2025-12-03T23:44:28Z
4,Underdog,player_points,Evan Mobley,Over,18.5,-137,2025-12-04,2025-12-03T23:44:28Z


In [7]:
from PRODUCTION.featureEngine.feature_engine import FeatureEngine

# Initialize FeatureEngine with NGBOOST model for points prediction
engine = FeatureEngine({
    "min_model": "../MODELS/SAVED_MODELS/min_model.pkl",
    "usg_model": "../MODELS/SAVED_MODELS/usg_model.pkl",
    "ngboost_model_paths": {
        "mean_model": "../MODELS/SAVED_MODELS/NGBOOST_PTS_MEAN_MODEL_PRODUCTION.pkl",
        "variance_model": "../MODELS/SAVED_MODELS/NGBOOST_PTS_VAR_MODEL_PRODUCTION.pkl",
        "calibration_factor": "../MODELS/SAVED_MODELS/NGBOOST_PTS_CALIBRATION_FACTOR_PRODUCTION.pkl",
        "calibration_params": "../MODELS/SAVED_MODELS/NGBOOST_PTS_CALIBRATION_PARAMS_PRODUCTION.pkl",  # Add this
        "features": "../MODELS/SAVED_MODELS/pts_features.pkl"
    }
})

# Test prediction for a single player
result = engine.project_player(
    player_name="Russell Westbrook",
    data=s26,
    date="2025-11-30",
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

print("Prediction Result:")
print(f"  Predicted Minutes: {result['predicted_minutes']:.2f}")
print(f"  Predicted Usage: {result['predicted_usage']:.3f}")
print(f"  Predicted Points: {result['predicted_points']:.2f}")
print(f"\nFull result: {result}")

Prediction Result:
  Predicted Minutes: 32.46
  Predicted Usage: 0.233
  Predicted Points: 21.81

Full result: {'predicted_minutes': 32.4560661315918, 'predicted_usage': 0.2327331304550171, 'predicted_points': 21.807560270249606}


## Top EVs for 2 leg bets

### Underdog picks

In [4]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

underdogPairs = calculate2LegBets(
    s26, dfsPTS, engine, current_date, 
    edge_threshold=4, stake=10, max_player_appearances=1, top_n=10,
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

underdogPairs = underdogPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2','ODDS 1', 'ODDS 2', 'PREDICTION 1', 'PREDICTION 2', 'PROB 1', 'PROB 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']]
underdogPairs.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogPairs.csv', index=False)
underdogPairs.head()

Pre-computing predictions for 93 players...
Processing 80 players...
Generated 3007 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,ODDS 1,ODDS 2,PREDICTION 1,PREDICTION 2,PROB 1,PROB 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV%,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
1545,Jeremy Sochan,Russell Westbrook,5.5,12.5,-115,-110,11.98,23.90,0.874,0.919,over,over,1,136.12,0.681,Med,High
1269,Ben Sheppard,Josh Okogie,5.5,6.5,-136,-110,10.52,13.43,0.837,0.894,over,over,1,119.96,0.600,Low,Med
2801,Steven Adams,Ausar Thompson,5.5,10.5,-112,-104,9.76,17.47,0.865,0.834,over,over,1,112.03,0.560,Low,High
2961,Ryan Rollins,Ryan Nembhard,13.5,8.5,-125,-120,21.57,14.78,0.827,0.809,over,over,1,96.73,0.484,High,High
2022,Onyeka Okongwu,Liam McNeeley,20.5,4.5,-112,-110,16.13,8.09,0.795,0.783,under,over,0,82.91,0.415,Med,Low


### Prizepicks picks

In [5]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points') & (dfsData['LINE'] < 24)]

prizepicksPairs = calculate2LegBets(
    s26, dfsPTS, engine, current_date, 
    edge_threshold=4, stake=10, max_player_appearances=1, top_n=10,
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

pairsPrizepicks = prizepicksPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'PROB 1', 'PROB 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
pairsPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksPairs.csv', index=False)
prizepicksPairs

Pre-computing predictions for 105 players...
Processing 97 players...
Generated 4417 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,ODDS 1,ODDS 2,PREDICTION 1,PREDICTION 2,MODEL SIDE 1,MODEL SIDE 2,PROB 1,PROB 2,PROB BOTH,EDGE 1,EDGE 2,COMBINED EDGE,EV%,KELLY FULL,RECOMMENDATION,SIGMA 1,SIGMA 2,SIGMA FLAG 1,SIGMA FLAG 2,CI 1,CI 2,CORRELATION,SAME_GAME,EXPECTED ROI
3878,Patrick Williams,Drew Eubanks,8.5,4.5,100,105,16.95,11.79,over,over,0.925,0.923,0.8369,0.437,0.447,0.603,151.07,0.755,1,5.55,4.94,Med,Low,"(6.1, 27.8)","(2.1, 21.5)",0.05,0,151.1
2639,Jeremy Sochan,Russell Westbrook,5.5,13.0,-115,-137,11.98,23.90,over,over,0.874,0.908,0.7784,0.352,0.344,0.483,133.52,0.668,1,5.50,7.85,Med,High,"(1.2, 22.7)","(8.5, 39.3)",0.05,0,133.5
2763,Jonathan Isaac,Steven Adams,2.5,5.5,-130,-112,6.24,9.76,over,over,0.846,0.865,0.7172,0.295,0.350,0.432,115.16,0.576,0,3.78,3.43,Low,Low,"(0.0, 13.7)","(3.0, 16.5)",0.05,0,115.2
1900,Ben Sheppard,Luke Kornet,5.5,6.5,-136,-132,10.52,11.69,over,over,0.837,0.839,0.6875,0.274,0.283,0.374,106.26,0.531,1,4.86,4.88,Low,Low,"(1.0, 20.0)","(2.1, 21.2)",0.05,0,106.3
4343,Ausar Thompson,Ryan Nembhard,10.5,8.5,-104,-120,17.47,14.78,over,over,0.834,0.809,0.6607,0.336,0.277,0.395,98.22,0.491,1,6.79,7.04,High,High,"(4.2, 30.8)","(1.0, 28.6)",0.05,0,98.2
3091,Kobe Brown,Ryan Rollins,4.0,14.5,-137,105,7.43,21.57,over,over,0.800,0.794,0.6220,0.236,0.318,0.353,86.61,0.433,0,3.80,8.13,Low,High,"(0.0, 14.9)","(5.6, 37.5)",0.05,0,86.6
3590,Liam McNeeley,Jalen Duren,4.5,17.5,-110,105,8.09,23.79,over,over,0.783,0.781,0.5991,0.272,0.305,0.355,79.72,0.399,0,4.46,7.49,Low,High,"(0.0, 16.8)","(9.1, 38.5)",0.05,0,79.7
2452,Julian Champagnie,Keegan Murray,9.5,16.5,-108,100,15.02,22.85,over,over,0.769,0.775,0.5837,0.262,0.287,0.335,75.12,0.376,1,7.34,7.78,High,High,"(0.6, 29.4)","(7.6, 38.1)",0.05,0,75.1
23,Darius Garland,Harrison Barnes,18.5,11.5,-113,-108,24.68,17.36,over,over,0.761,0.768,0.5727,0.244,0.261,0.309,71.82,0.359,1,8.02,7.62,High,High,"(9.0, 40.4)","(2.4, 32.3)",0.05,0,71.8
3800,Tyrese Martin,Zach LaVine,9.5,18.5,100,-105,14.89,24.05,over,over,0.760,0.760,0.5656,0.272,0.260,0.320,69.67,0.348,1,7.55,7.17,High,High,"(0.1, 29.7)","(10.0, 38.1)",0.05,0,69.7


## 3 leg parlay

### Underdog picks

In [6]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points') ]

underdogTrios = calculate3LegBets(
    s26, dfsPTS, engine, current_date, 
    edge_threshold=4, stake=10, max_player_appearances=1, top_n=10,
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

underdogTrios = underdogTrios[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'PROB 1', 'PROB 2', 'PROB 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
underdogTrios.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogTrios.csv', index=False)
underdogTrios.head()

Pre-computing predictions for 93 players...
Processing 80 players...
Generated 70548 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,PROB 1,PROB 2,PROB 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV%,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
39541,Ben Sheppard,Jeremy Sochan,Russell Westbrook,5.5,5.5,12.5,10.52,11.98,23.90,0.837,0.874,0.919,over,over,over,1,262.86,0.526,Low,Med,High
69964,Josh Okogie,Ausar Thompson,Ryan Nembhard,6.5,10.5,8.5,13.43,17.47,14.78,0.894,0.834,0.809,over,over,over,1,225.56,0.451,Med,High,High
58728,Onyeka Okongwu,Steven Adams,Ryan Rollins,20.5,5.5,13.5,16.13,9.76,21.57,0.795,0.865,0.827,under,over,over,1,207.18,0.414,Med,Low,High
66049,Liam McNeeley,Alperen Sengun,Jalen Duren,4.5,22.5,17.5,8.09,27.87,23.79,0.783,0.780,0.781,over,over,over,0,157.44,0.315,Low,High,High
3637,Darius Garland,Julian Champagnie,Keegan Murray,18.5,9.5,16.5,24.68,15.02,22.85,0.761,0.769,0.775,over,over,over,1,144.85,0.290,High,High,High


### Prizepicks picks

In [8]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points') & (dfsData['LINE'] < 24) & (dfsData['LINE'] > 5)]

triosPrizepicks = calculate3LegBets(
    s26, dfsPTS, engine, current_date, 
    edge_threshold=4, stake=10, max_player_appearances=1, top_n=10,
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

triosPrizepicks = triosPrizepicks[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'PROB 1', 'PROB 2', 'PROB 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
triosPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

Pre-computing predictions for 100 players...
Processing 92 players...
Generated 106962 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,PROB 1,PROB 2,PROB 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV%,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
83666,Jeremy Sochan,Patrick Williams,Russell Westbrook,5.5,8.5,13.0,11.98,16.95,23.90,0.874,0.925,0.908,over,over,over,1,296.75,0.594,Med,Med,High
64771,Ben Sheppard,Luke Kornet,Steven Adams,5.5,6.5,5.5,10.52,11.69,9.76,0.837,0.839,0.865,over,over,over,1,227.73,0.455,Low,Low,Low
105206,Keegan Murray,Ausar Thompson,Ryan Nembhard,16.5,10.5,8.5,22.85,17.47,14.78,0.775,0.834,0.809,over,over,over,1,182.17,0.364,High,High,High
1712,Darius Garland,Julian Champagnie,Ryan Rollins,18.5,9.5,14.5,24.68,15.02,21.57,0.761,0.769,0.794,over,over,over,1,150.74,0.301,High,High,High
77725,Harrison Barnes,Zach LaVine,Jalen Duren,11.5,18.5,17.5,17.36,24.05,23.79,0.768,0.760,0.781,over,over,over,1,145.94,0.292,High,High,High
